# 09 - Detector de deslizamiento mediante Aprendizaje con Instancias Múltiples (MIL) y W&B

Este entorno se destina al entrenamiento de la rama algorítmica específica para la anomalía de deslizamiento, implementando un paradigma de aprendizaje débilmente supervisado (Multiple Instance Learning o MIL) estructurado por ciclos o vueltas.

El enfoque conceptual difiere sustancialmente del propuesto en el cuaderno 08 (CNN supervisada tradicional): en este caso, se omite el requerimiento de determinar el instante exacto del evento de patinaje. En su lugar, cada ciclo que transcurre por el segmento de la vía con anomalía se define como una "bolsa positiva", mientras que los ciclos correspondientes a series prolongadas de operación normal actúan como "bolsas negativas".

La lógica de entrenamiento subyacente se ha consolidado en el archivo `herramientas/train_slip_mil_cnn.py`, permitiendo su ejecución de manera reproducible desde la interfaz de comandos y garantizando el trazado integral del proceso en la plataforma W&B.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def find_inference_root():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'app').is_dir() and (candidate / 'tools').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('No se ha encontrado el directorio raíz de inference_server.')

INFERENCE_ROOT = find_inference_root()
TFM_ROOT = INFERENCE_ROOT.parent
print(f'INFERENCE_ROOT={INFERENCE_ROOT}')
print(f'TFM_ROOT={TFM_ROOT}')

## Configuración

Los parámetros de configuración del modelo se encuentran centralizados en el archivo `../config/slip_mil_experiments.json`. Para evaluar arquitecturas o parámetros alternativos sin alterar el código de este cuaderno, se debe modificar la variable `active_experiment` dentro de dicho fichero JSON.

In [ ]:
CONFIG_PATH = INFERENCE_ROOT / 'config' / 'slip_mil_experiments.json'
config_doc = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
EXPERIMENT_NAME = config_doc['active_experiment']
defaults = config_doc['defaults']
experiment = {**defaults, **config_doc['experiments'][EXPERIMENT_NAME]}
wandb_cfg = config_doc['wandb']

print(f'Experimento activo: {EXPERIMENT_NAME}')
print(json.dumps(experiment, indent=2, ensure_ascii=False))

## Histórico de series de captura relevantes

A continuación se detallan las ejecuciones y modelos previos que proporcionan el marco contextual para las comparativas analíticas en W&B. Esta lista se integra en el archivo JSON con el objetivo de asegurar su control de versiones en conjunción con la configuración del experimento.

In [ ]:
for item in config_doc.get('previous_wandb_runs', []):
    print(f"- {item['name']}: {item['role']}")

## Conjuntos de datos empleados

El proceso de entrenamiento registra en W&B un artefacto denominado `dataset`, el cual encapsula los archivos CSV utilizados junto con el manifiesto descriptivo del conjunto de datos. Esta práctica garantiza una trazabilidad estricta, vinculando cada ejecución del modelo con los datos precisos que sustentaron su entrenamiento.

In [ ]:
slip_csvs = experiment.get('slip_csvs') or [experiment.get('slip_csv')]
dataset_paths = {
    'slip_csvs': [(INFERENCE_ROOT / path).resolve() for path in slip_csvs],
    'normal_csv': (INFERENCE_ROOT / experiment['normal_csv']).resolve(),
    'dataset_manifest': TFM_ROOT / 'datos' / 'DATASET_MANIFEST.md',
}
for name, value in dataset_paths.items():
    paths = value if isinstance(value, list) else [value]
    for path in paths:
        print(f'{name}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')
        assert path.exists(), f'El archivo o directorio no existe: {path}'


## Ejecución del entrenamiento MIL

En esta celda se invoca el script de entrenamiento parametrizado mediante el archivo JSON. Condicionado por el parámetro `wandb.enabled=true`, el proceso sincronizará la ejecución con W&B, almacenando el modelo resultante de forma concurrente al artefacto del conjunto de datos.

In [ ]:
slip_csvs = experiment.get('slip_csvs') or [experiment.get('slip_csv')]
command = [
    sys.executable,
    str(INFERENCE_ROOT / 'tools' / 'train_slip_mil_cnn.py'),
    '--model-name', experiment['model_name'],
    '--normal-csv', experiment['normal_csv'],
    '--slip-end-s', str(experiment['slip_end_s']),
    '--lap-period-s', str(experiment['lap_period_s']),
    '--window-step', str(experiment['window_step']),
    '--normal-max-bags', str(experiment['normal_max_bags']),
    '--normal-train-fraction', str(experiment['normal_train_fraction']),
    '--max-normal-fp-rate', str(experiment['max_normal_fp_rate']),
    '--epochs', str(experiment['epochs']),
    '--learning-rate', str(experiment['learning_rate']),
    '--weight-decay', str(experiment['weight_decay']),
    '--dropout', str(experiment['dropout']),
    '--seed', str(experiment['seed']),
    '--output-dir', experiment['output_dir'],
    '--models-dir', experiment['models_dir'],
    '--wandb-project', wandb_cfg['project'],
    '--wandb-entity', wandb_cfg['entity'],
]
for slip_csv in slip_csvs:
    command.extend(['--slip-csv', slip_csv])
if wandb_cfg.get('enabled', False):
    command.append('--wandb')

print(' '.join(command))
subprocess.run(command, cwd=INFERENCE_ROOT, check=True)


## Resumen analítico generado

Se procede a la ingesta del archivo JSON local emitido por el script de entrenamiento, con el fin de exponer las métricas de desempeño fundamentales de manera directa dentro del entorno del cuaderno.

In [ ]:
summary_path = (INFERENCE_ROOT / experiment['output_dir'] / f"{experiment['model_name']}_summary.json").resolve()
summary_doc = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary_doc['summary'], indent=2, ensure_ascii=False))

## Evaluación preliminar

Se procede a evaluar el modelo entrenado empleando tanto el conjunto de datos de tipo mixto (slip/mixed) como la serie de operación normal de larga duración. Esta validación asume un carácter crítico: un modelo especializado en deslizamiento (slip) dotado de alta sensibilidad pierde su viabilidad técnica si adolece de una tasa inaceptable de falsos positivos frente a condiciones nominales.

In [ ]:
model_path = (INFERENCE_ROOT / experiment['models_dir'] / f"{experiment['model_name']}.pth").resolve()
eval_dir = (INFERENCE_ROOT / experiment['output_dir']).resolve()
slip_csvs = experiment.get('slip_csvs') or [experiment.get('slip_csv')]
eval_commands = [
    [
        sys.executable,
        str(INFERENCE_ROOT / 'tools' / 'evaluate_multiscale_slip_live_csv.py'),
        slip_csvs[-1],
        '--model-path', str(model_path),
        '--split-s', str(experiment['slip_end_s']),
        '--output-csv', str(eval_dir / f"{experiment['model_name']}_eval_latest_slip.csv"),
    ],
    [
        sys.executable,
        str(INFERENCE_ROOT / 'tools' / 'evaluate_multiscale_slip_live_csv.py'),
        experiment['normal_csv'],
        '--model-path', str(model_path),
        '--split-s', '999999',
        '--output-csv', str(eval_dir / f"{experiment['model_name']}_eval_normal.csv"),
    ],
]

for eval_command in eval_commands:
    print('\n$ ' + ' '.join(eval_command))
    subprocess.run(eval_command, cwd=INFERENCE_ROOT, check=True)
